# accel-sim silicon anchor — GEMM-size isolation

The chain-generalize anchor found that a single large, well-occupied
2048×2048 GEMM (depth=1, no chaining at all) runs close to the model's
`compute_efficiency=1.0` ideal, while GPT-2's own 768×768 shape needed the
full 0.55 discount plus an occupancy fix. That comparison mixed GEMM
**size** with **occupancy** (768×768 is occupancy-penalized under
`util_tiles=128`, 2048×2048 is not).

This notebook isolates size alone: depth is fixed at 1 (a single GEMM, no
chaining) and tokens fixed at 8192 (GPT-2's own count), while a square
layer's size (K=N) sweeps from heavily occupancy-penalized (256) through
the occupancy threshold (crossed around 1536–2048) up past it (4096). The
resulting curve shows whether there's a real, smooth size-dependent
efficiency effect distinct from the existing occupancy penalty.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then
`Runtime > Run all`.

Writes `gemm_size_profile.json`, prints it, and auto-downloads it. Bring
that file back and run:

```bash
python validate/silicon/compare_gemm_size.py gemm_size_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
ITERS  = 50
WARMUP = 15
OUT    = "gemm_size_profile.json"

TOKENS = 8 * 1024
SIZES = [256, 512, 768, 1024, 1536, 2048, 3072, 4096]   # square (K=N) layers


In [ ]:
import torch
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
dtype = torch.float16
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu}   dtype=fp16   tokens={TOKENS}   depth=1   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics
import torch.nn as nn

def bench(size, M, iters, warmup):
    lin = nn.Linear(size, size, bias=False).to(device=device, dtype=dtype)
    opt = torch.optim.Adam(lin.parameters(), lr=1e-4)
    x = torch.randn(M, size, device=device, dtype=dtype, requires_grad=True)

    fwd, bwd, optt = [], [], []
    for i in range(warmup + iters):
        ev = [torch.cuda.Event(enable_timing=True) for _ in range(5)]
        ev[0].record()
        y = lin(x)
        ev[1].record()
        loss = y.float().square().mean()
        ev[2].record()
        loss.backward()
        ev[3].record()
        opt.step()
        opt.zero_grad(set_to_none=True)
        x.grad = None
        ev[4].record()
        torch.cuda.synchronize()
        if i >= warmup:
            fwd.append(ev[0].elapsed_time(ev[2]))
            bwd.append(ev[2].elapsed_time(ev[3]))
            optt.append(ev[3].elapsed_time(ev[4]))

    def stat(v):
        return {"mean_ms": statistics.fmean(v),
                "std_ms": statistics.pstdev(v) if len(v) > 1 else 0.0}
    return {"forward": stat(fwd), "backward": stat(bwd), "optimizer": stat(optt)}


In [ ]:
results = {}
for size in SIZES:
    r = bench(size, TOKENS, ITERS, WARMUP)
    results[str(size)] = r
    print(f"  size {size:5d}  fwd {r['forward']['mean_ms']:8.3f}  "
          f"bwd {r['backward']['mean_ms']:8.3f}  "
          f"opt {r['optimizer']['mean_ms']:6.3f} ms")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "dtype": "float16", "tokens": TOKENS, "sizes": SIZES,
    "iters": ITERS, "warmup": WARMUP,
    "results": results, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
